In [1]:
import pandas as pd

In [2]:
path = "C:/Users/vihaa/Desktop/Solar Panel/Solar Panel Fault Detection"

path = r"C:\Users\vihaa\Desktop\Solar Panel\Solar Panel Fault Detection
unicodeescape

In [3]:
solar = pd.read_csv(path + "/backend/data/timeseries/raw/Solar_Energy_Generation.csv")
weather=pd.read_csv(path+"/backend/data/timeseries/raw/Weather_Data_reordered_all.csv")

In [4]:
print(solar.head())
solar.shape

   CampusKey  SiteKey            Timestamp  SolarGeneration
0          2        1  2020-01-01 00:15:00              NaN
1          2        1  2020-01-01 00:30:00              NaN
2          2        1  2020-01-01 00:45:00              NaN
3          2        1  2020-01-01 01:00:00              NaN
4          2        1  2020-01-01 01:15:00              NaN


(2731946, 4)

In [5]:
print(weather.head())
weather.shape

   CampusKey            Timestamp  ApparentTemperature  AirTemperature  \
0          1  2020-01-01 00:00:00            13.666667       13.880000   
1          1  2020-01-01 00:15:00            13.206667       13.666667   
2          1  2020-01-01 00:30:00            12.840000       13.553333   
3          1  2020-01-01 00:45:00            12.113333       13.506667   
4          1  2020-01-01 01:00:00            11.946667       13.260000   

   DewPointTemperature  RelativeHumidity  WindSpeed  WindDirection  
0             8.960000         72.400000   0.000000     188.133333  
1             9.040000         73.466667   1.200000     203.866667  
2             9.053333         74.000000   2.520000     222.800000  
3             9.100000         74.466667   5.986667     231.133333  
4             9.266667         76.533333   5.946667     247.866667  


(371769, 8)

Now I want to visualise the Solar Dataset
1)Which Site and which Panels wrt What Time produce Solar?

In [6]:
solar['Timestamp']=pd.to_datetime(solar['Timestamp'])

No need to visualise since I have understood the dataset

Kaggle #Summary is helpful to understand the data

Next step:- Clean the data so that we can merge the datasets(which I believe we should do)
and thereafter train the model(easier part)

Solution:- Identify problems in both the datasets

For now only:- Site 1
Merge on Weather Data?


In [7]:
site1=solar[solar["SiteKey"]==1]
#Time series is odered


In [8]:
site1.isna().sum()

CampusKey              0
SiteKey                0
Timestamp              0
SolarGeneration    41746
dtype: int64

In [9]:
(site1['SolarGeneration'].isna().sum() / len(site1)) * 100

np.float64(52.6305172783318)

In [10]:
counts = site1.groupby(site1['Timestamp'].dt.hour)['SolarGeneration'].agg(
    total='size',
    nan_count=lambda x: x.isna().sum(),
    not_nan_count=lambda x: x.notna().sum()
)

In [11]:
print(counts)

           total  nan_count  not_nan_count
Timestamp                                 
0           3308       3308              0
1           3320       3320              0
2           3321       3321              0
3           3323       3323              0
4           3321       3321              0
5           3316       3316              0
6           3316       2730            586
7           3315        987           2328
8           3310        115           3195
9           3308         84           3224
10          3308         68           3240
11          3308        143           3165
12          3305        315           2990
13          3301        362           2939
14          3300        222           3078
15          3300         73           3227
16          3302         53           3249
17          3301        657           2644
18          3296       1471           1825
19          3296       1813           1483
20          3296       2896            400
21         

We have to diff when solar generation is 0 and Null

In [12]:
counts = (
    site1.assign(
        month=site1['Timestamp'].dt.month,
        hour=site1['Timestamp'].dt.hour
    )
    .groupby(['month','hour'])['SolarGeneration']
    .agg(
        total='size',
        nan_count=lambda x: x.isna().sum(),
        not_nan_count=lambda x: x.notna().sum()
    )
)

In [13]:
#pd.set_option('display.max_rows', None)
print(counts)

            total  nan_count  not_nan_count
month hour                                 
1     0       371        371              0
      1       372        372              0
      2       372        372              0
      3       372        372              0
      4       372        372              0
...           ...        ...            ...
12    19      240          3            237
      20      240        123            117
      21      239        239              0
      22      236        236              0
      23      236        236              0

[288 rows x 3 columns]


Month	Sunrise	Sunset	Day Length
Jan	earliest	latest	longest
Feb	slightly later	slightly earlier	↓
Mar	increasing	decreasing	medium
Apr	later	earlier	↓
May	later	earlier	↓
Jun	latest	earliest	shortest
Jul	slightly earlier	slightly later	↑
Aug	earlier	later	↑
Sep	balanced	balanced	medium
Oct	earlier	later	↑
Nov	early	late	↑
Dec	earliest	latest	longest

In [14]:
site1 = site1.copy()

In [15]:
time_diff = site1['Timestamp'].sort_values().diff()

print(time_diff.value_counts())

Timestamp
0 days 00:15:00    79303
0 days 22:00:00        1
1 days 06:30:00        1
0 days 09:00:00        1
2 days 01:30:00        1
4 days 00:15:00        1
2 days 02:45:00        1
0 days 19:30:00        1
1 days 03:30:00        1
0 days 01:45:00        1
0 days 11:15:00        1
0 days 12:15:00        1
1 days 03:00:00        1
1 days 16:00:00        1
0 days 01:15:00        1
1 days 07:15:00        1
Name: count, dtype: int64


In [19]:
site1 = site1.set_index('Timestamp').sort_index()
site1 = site1.asfreq('15T')

KeyError: "None of ['Timestamp'] are in the columns"

In [23]:
site1 = site1[
    (site1.index.hour >= 6) &
    (site1.index.hour <= 20)
]

In [24]:
site1

,CampusKey,SiteKey,SolarGeneration
Timestamp,,,
2020-01-01 06:00:00,2.0,1.0,NaN
2020-01-01 06:15:00,2.0,1.0,0.135
2020-01-01 06:30:00,2.0,1.0,0.465
2020-01-01 06:45:00,2.0,1.0,1.039
2020-01-01 07:00:00,2.0,1.0,1.673
...,...,...,...
2022-04-23 19:45:00,2.0,1.0,NaN
2022-04-23 20:00:00,2.0,1.0,NaN
2022-04-23 20:15:00,2.0,1.0,NaN


In [25]:
import numpy as np

def process_day(df):
    df = df.copy()
    
    values = df['SolarGeneration'].values
    
    # Find indices where generation > 0
    valid_idx = np.where(values > 0)[0]
    
    # If no valid values → keep all as 0
    if len(valid_idx) == 0:
        return df
    
    first_valid = valid_idx[0]
    last_valid = valid_idx[-1]

    df.iloc[:first_valid, df.columns.get_loc('SolarGeneration')] = 0
    
    # ✅ FIX 2: after last valid → set to 0
    df.iloc[last_valid+1:, df.columns.get_loc('SolarGeneration')] = 0
    
    # Middle part (where interpolation should happen)
    middle = df.iloc[first_valid:last_valid+1].copy()
    
    # Replace zeros with NaN for interpolation
    middle['SolarGeneration'] = middle['SolarGeneration'].replace(0, np.nan)
    
    # Interpolate
    middle['SolarGeneration'] = middle['SolarGeneration'].interpolate(method='linear')
    
    # Put back
    df.iloc[first_valid:last_valid+1] = middle
    
    return df

In [26]:
site1 = site1.groupby(site1.index.date).apply(process_day).reset_index(drop=True)

In [27]:
site1.isna().sum()

CampusKey          1078
SiteKey            1078
SolarGeneration     900
dtype: int64

In [28]:
site1

,CampusKey,SiteKey,SolarGeneration
0,2.0,1.0,0.000
1,2.0,1.0,0.135
2,2.0,1.0,0.465
3,2.0,1.0,1.039
4,2.0,1.0,1.673
...,...,...,...
50635,2.0,1.0,0.000
50636,2.0,1.0,0.000
50637,2.0,1.0,0.000
50638,2.0,1.0,0.000


Why this happened

You likely already did:

site1 = site1.set_index('Timestamp')

So now:

'Timestamp' is not a column anymore ❌
It became the index ✅

In [33]:
site1['hour'] = site1.index.hour
site1['day'] = site1.index.day
site1['month'] = site1.index.month
site1['day_of_week'] = site1.index.dayofweek

AttributeError: 'RangeIndex' object has no attribute 'hour'

In [35]:
site1

,CampusKey,SiteKey,SolarGeneration,hour,day,month,day_of_week
Timestamp,,,,,,,
2020-01-01 06:00:00,2,1,0.000,6,1,1,2
2020-01-01 06:15:00,2,1,0.135,6,1,1,2
2020-01-01 06:30:00,2,1,0.465,6,1,1,2
2020-01-01 06:45:00,2,1,1.039,6,1,1,2
2020-01-01 07:00:00,2,1,1.673,7,1,1,2
...,...,...,...,...,...,...,...
2022-04-23 19:45:00,2,1,0.000,19,23,4,5
2022-04-23 20:00:00,2,1,0.000,20,23,4,5
2022-04-23 20:15:00,2,1,0.000,20,23,4,5


In [37]:
time_diff = site1.index.to_series().diff()
print(time_diff.value_counts())

Timestamp
0 days 00:15:00    48732
0 days 09:15:00      816
1 days 09:15:00        4
2 days 09:15:00        2
1 days 12:15:00        1
0 days 09:00:00        1
4 days 09:15:00        1
0 days 17:00:00        1
0 days 18:00:00        1
1 days 21:45:00        1
1 days 13:00:00        1
Name: count, dtype: int64


Fix and check if time series dataset is complete wrt time intervals and similiarly weather data
Actually do not remove 0 from night time
and merge with weather data
and then see if there is any other pre processing left
tmr from 0 to hero nderstand what you is doong
and thenfix the preporcessing and complete it
and then finally run the model